In [2]:
import os

from typing import Annotated, Optional

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

from typing_extensions import TypedDict

## ENVIRONMENT

In [3]:
load_dotenv()

True

## LLM

In [4]:
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0,
    api_key=os.getenv("OPENAI_API_KEY")
)

## STATE

In [6]:
class State(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

    # Human feedback will be stored separately
    human_feedback: Optional[str]

## CHATBOT NODE

In [7]:
def chatbot(state: State):
    """
        Generate an AI response using the complete
        conversation history.
    """

    print("\n CHATBOT NODE")

    response = llm.invoke(
        state["messages"]
    )

    return{
        "messages" : [response]
    }
    

## HUMAN FEEDBACK NODE

In [9]:
def human_feedback(state: State):
    """
    Ask a human to review the AI response.

    interrupt() pauses the graph.

    The graph will continue when we send:

        Cmmand(resume="yes")
    """

    # Get AI's latest response
    ai_response = state['messages'][-1].content

    print("\n")
    print("=" * 70)
    print("🤖 AI RESPONSE")
    print("=" * 70)

    print(ai_response)

    print("\n⏸️ Waiting for human feedback...")

    # INTERRUPT

    feedback = interrupt(
        {
            "question" : "Do you approve this response?",
            "ai_response" : ai_response,
            "option" : [
                "yes",
                "No",
                "edit"
            ]
        }
    )

    # GRAPH RESUMES HERE

    print(
        f"\n👤 Human feedback: {feedback}"
    )

    return {
        "human_feedback": feedback
    }


## CREATE GRAPH

In [11]:
builder = StateGraph(State)

# ADD nodes

builder.add_node(
    "chatbot",
    chatbot
)

builder.add_node(
    "human_feedback",
    human_feedback
)

## GRAPH EDGES

In [13]:
builder.add_edge(
    START,
    "chatbot"
)

builder.add_edge(
    "chatbot",
    "human_feedback"
)

builder.add_edge(
    "human_feedback",
    END
)

## MEMORY

In [15]:
memory = InMemorySaver()

# Compile graph

graph = builder.compile(
    checkpointer=memory
)


## THREAD CONFIGURATION

In [16]:
config = {
    "configurable" : {
        "thread_id" : "1"
    }
}

## SEND MESSAGE FUNCTION

In [17]:
def send_message(user_message : str):
    """
    Send a message to the LangGraph chatbot.
    """

    print("\n")
    print("=" * 70)
    print(f"👤 USER: {user_message}")
    print("=" * 70)

    input_data = {
        "messages" :[
            HumanMessage(
                content = user_message
            )
        ]
    }

    # STREAM GRAPH EXECUTION

    for event in graph.stream( input_data, config = config, stream_mode = "values"):

        # Check if messages are available

        if "messages" not in event:
            continue

        messages = event["messages"]

        if not messages:
            continue

        latest_message = messages[-1]

        print(f"\n📨 {latest_message.content}")


## FIRST CONVERSATION

In [18]:
print("\n")
print("🚀 LANGGRAPH CHATBOT")
print("=" * 70)

send_message(
    "Hi! My name is Ayush."
)



🚀 LANGGRAPH CHATBOT


👤 USER: Hi! My name is Ayush.

📨 Hi! My name is Ayush.

 CHATBOT NODE

📨 Hi Ayush! How can I assist you today?


🤖 AI RESPONSE
Hi Ayush! How can I assist you today?

⏸️ Waiting for human feedback...

📨 Hi Ayush! How can I assist you today?


## HUMAN FEEDBACK

In [19]:
print("\n")
print("=" * 70)
print("⏸️ HUMAN FEEDBACK REQUIRED")
print("=" * 70)

feedback = input(
    "\nEnter your feedback "
    "(yes/no/edit): "
)

# Resume the interrupted graph

for event in graph.stream( Command( resume = feedback), config = config, stream_mode = "values"):

    if "messages" not in event:
        continue

    messages = event["messages"]

    if messages :

        latest_message = messages[-1]

        print(f"\n📨 {latest_message.content}")



⏸️ HUMAN FEEDBACK REQUIRED

📨 Hi Ayush! How can I assist you today?


🤖 AI RESPONSE
Hi Ayush! How can I assist you today?

⏸️ Waiting for human feedback...

👤 Human feedback: edit

📨 Hi Ayush! How can I assist you today?


## TEST MEMORY

print("\n")
print("=" * 70)
print("🧠 MEMORY TEST")
print("=" * 70)

send_message(
    "What is my name?"
)

## END

In [20]:
print("\n")
print("=" * 70)
print("✅ PROGRAM COMPLETED")
print("=" * 70)



✅ PROGRAM COMPLETED
